In [3]:
# Colab-compatible random shelf assignment tool
# ----------------------------------------------
# Requirements: pandas + openpyxl (Colab already has them)
# It will prompt you to upload your Excel file, randomize Shelf/Row/Column,
# and give you a download link for the modified version.

import pandas as pd
import random
from google.colab import files
import io

# ==== Config ====
SHELF_BANKS = ['A','B','C']
ROWS = list(range(1, 3))
COLUMNS = list(range(1, 4))
TOTAL_SLOTS = len(SHELF_BANKS) * len(ROWS) * len(COLUMNS)
SHELF_COL_NAME, ROW_COL_NAME, COLUMN_COL_NAME = 'Shelf', 'Row', 'Column'

# ==== Helper functions ====
def generate_all_slots():
    return [(shelf, row, col) for shelf in SHELF_BANKS for row in ROWS for col in COLUMNS]

def assign_slots_to_items(n_items):
    all_slots = generate_all_slots()
    random.shuffle(all_slots)
    if n_items <= TOTAL_SLOTS:
        return all_slots[:n_items]
    else:
        return all_slots

# ==== File upload ====
print("📂 Please upload your Excel file (.xlsx)")
uploaded = files.upload()

if not uploaded:
    raise SystemExit("No file uploaded.")

filename = list(uploaded.keys())[0]
print(f"Uploaded: {filename}")

# ==== Read workbook ====
sheets = pd.ExcelFile(io.BytesIO(uploaded[filename]), engine='openpyxl').sheet_names
print("Sheets found:", ", ".join(sheets))
sheet_name = input("Enter the sheet name to modify: ").strip()

# ==== Load selected sheet ====
df = pd.read_excel(io.BytesIO(uploaded[filename]), sheet_name=sheet_name, engine='openpyxl')
print(f"Loaded {len(df)} rows from sheet '{sheet_name}'")

# Normalize column names
col_map = {}
for col in df.columns:
    if col.lower() == SHELF_COL_NAME.lower():
        col_map[col] = SHELF_COL_NAME
    elif col.lower() == ROW_COL_NAME.lower():
        col_map[col] = ROW_COL_NAME
    elif col.lower() == COLUMN_COL_NAME.lower():
        col_map[col] = COLUMN_COL_NAME
df = df.rename(columns=col_map)
for c in [SHELF_COL_NAME, ROW_COL_NAME, COLUMN_COL_NAME]:
    if c not in df.columns:
        df[c] = pd.NA

# ==== Random assignment ====
n_items = len(df)
slots = assign_slots_to_items(n_items)
random.seed(None)
row_indices = list(df.index)
random.shuffle(row_indices)

for i in range(min(n_items, len(slots))):
    idx = row_indices[i]
    shelf, row, col = slots[i]
    df.at[idx, SHELF_COL_NAME] = shelf
    df.at[idx, ROW_COL_NAME] = row
    df.at[idx, COLUMN_COL_NAME] = col

if n_items > len(slots):
    print(f"⚠️ {n_items - len(slots)} items could not be assigned (marked UNASSIGNED)")
    for i in range(len(slots), n_items):
        idx = row_indices[i]
        df.at[idx, SHELF_COL_NAME] = "UNASSIGNED"
        df.at[idx, ROW_COL_NAME] = pd.NA
        df.at[idx, COLUMN_COL_NAME] = pd.NA

# ==== Save modified workbook ====
out_name = f"randomized_{filename}"
with pd.ExcelWriter(out_name, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"✅ Randomization complete. Download your file below:")
files.download(out_name)


📂 Please upload your Excel file (.xlsx)


Saving Test_Objects_List.xlsx to Test_Objects_List (2).xlsx
Uploaded: Test_Objects_List (2).xlsx
Sheets found: Sheet1
Enter the sheet name to modify: Sheet1
Loaded 18 rows from sheet 'Sheet1'
✅ Randomization complete. Download your file below:


/tmp/ipykernel_22179/2178661859.py:74: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'B' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, SHELF_COL_NAME] = shelf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>